<div align="center">

# ⚛️  Diatomic Pair MLP — From Scratch to Trained Model

### *A step-by-step tutorial for predicting energy & forces of a diatomic molecule*

</div>

---

**Goal:** We want a machine-learning model that predicts the **total energy** and
**atomic forces** of a two-atom system (e.g. O₂ or Br₂) as a function of bond length.

**What makes a diatomic system special:**

| Property | Value |
|---|---|
| Number of atoms | 2 |
| Number of unique distances | 1 (the bond length) |
| Symmetry | E(r) = E(r), F₀ = −F₁ |
| Ground truth | Morse potential (analytic) |

Because the whole physics reduces to a **1D function** E(r), a simple MLP is a
perfect fit — and it lets us focus entirely on the GOAL architecture.

**What you will learn:**

| # | Topic |
|---|-------|
| 1 | Generate a synthetic dataset using the Morse potential |
| 2 | Build `AtomicGraph` objects by hand |
| 3 | Write a **monolithic** `PairMLPModel` (the simplest GOAL model) |
| 4 | Write a **custom head** `PairEnergyForcesHead` that plugs into any backbone |
| 5 | Train with GOAL's `CompositeLoss` (energy + forces) |
| 6 | Evaluate: compare predicted vs. true energy curves |

> **Key insight:** Forces are obtained as the **negative gradient of the energy
> with respect to positions** (`-∂E/∂r`).  This is *autograd forces* — the
> gold standard for energy-conserving force fields.

---

## 1 · Imports

Everything we need from GOAL plus a few standard libraries.

In [ ]:
from __future__ import annotations

import math
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch_geometric.data import Batch
from torch_geometric.utils import scatter

# GOAL: data contract
from goal.ml.data.graph import AtomicGraph, NodeFeatures

# GOAL: neural-network building blocks
from goal.ml.nn.blocks.embedding import AtomicNumberEmbedding
from goal.ml.nn.primitives.radial import BesselBasis

# GOAL: loss system
from goal.ml.training.loss import (
    CompositeLoss,
    EnergyLoss,
    ForcesLoss,
    WeightedLoss,
)

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---

## 2 · The Physics: Morse Potential for O₂

We use the **Morse potential** as our ground-truth energy function:

$$E(r) = D_e \left(1 - e^{-\alpha (r - r_e)}\right)^2 - D_e$$

where:
- $D_e$ = dissociation energy (depth of the potential well)
- $r_e$ = equilibrium bond length
- $\alpha$ = controls the width of the potential well

The analytic force on each atom is $F = -dE/dr$ along the bond direction.

**O₂ parameters** (in eV and Angstrom):

In [ ]:
# O2 Morse parameters (from spectroscopic data)
D_e   = 5.16   # eV  — dissociation energy
alpha = 2.65   # Å⁻¹ — well width parameter
r_e   = 1.21   # Å   — equilibrium bond length

# Atomic number of oxygen
Z_O = 8


def morse_energy(r: np.ndarray) -> np.ndarray:
    """Morse potential energy (eV) for O2 at bond length r (Angstrom)."""
    return D_e * (1.0 - np.exp(-alpha * (r - r_e))) ** 2 - D_e


def morse_force_magnitude(r: np.ndarray) -> np.ndarray:
    """Magnitude of the force |F| = |dE/dr| for the Morse potential.

    Positive = repulsive (pushes atoms apart), negative = attractive.
    Force on atom 0 along bond direction = +dE/dr.
    Force on atom 1 along bond direction = -dE/dr.
    """
    return 2.0 * D_e * alpha * np.exp(-alpha * (r - r_e)) * (
        1.0 - np.exp(-alpha * (r - r_e))
    )


# Quick sanity check
r_check = np.array([r_e])
print(f"E(r_e) = {morse_energy(r_check)[0]:.4f} eV  (should be -{D_e} eV)")
print(f"F(r_e) = {morse_force_magnitude(r_check)[0]:.6f}  (should be 0 at equilibrium)")

# Plot the Morse potential
r_plot = np.linspace(0.8, 4.0, 400)
E_plot = morse_energy(r_plot)
F_plot = morse_force_magnitude(r_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(r_plot, E_plot, "C0", lw=2)
axes[0].axvline(r_e, ls="--", color="gray", label=f"$r_e = {r_e}$ Å")
axes[0].axhline(-D_e, ls=":", color="gray", label=f"$-D_e = {-D_e}$ eV")
axes[0].set_xlabel("Bond length $r$ (Å)")
axes[0].set_ylabel("Energy (eV)")
axes[0].set_title("Morse Potential — O₂")
axes[0].set_ylim(-6, 4)
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(r_plot, F_plot, "C1", lw=2)
axes[1].axvline(r_e, ls="--", color="gray", label=f"$r_e = {r_e}$ Å")
axes[1].axhline(0, ls="-", color="k", lw=0.8)
axes[1].set_xlabel("Bond length $r$ (Å)")
axes[1].set_ylabel("Force component $dE/dr$ (eV/Å)")
axes[1].set_title("Force along bond — O₂")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 3 · Build the Dataset

### 3.1 Generate structures

We sample bond lengths uniformly in the range [0.85, 3.5] Å.
Each structure is placed along the x-axis:

```
Atom 0 at (0, 0, 0)    Atom 1 at (r, 0, 0)
```

The forces follow from Newton's third law:

$$\mathbf{F}_0 = \left(\frac{dE}{dr}, 0, 0\right), \quad \mathbf{F}_1 = \left(-\frac{dE}{dr}, 0, 0\right)$$

In [ ]:
def make_o2_graph(r: float, dtype: torch.dtype = torch.float32) -> AtomicGraph:
    """Create an AtomicGraph for an O2 molecule at bond length r (Angstrom).

    Atom 0 is fixed at the origin; atom 1 is placed at (r, 0, 0).
    Energy and forces are computed from the Morse potential.

    Parameters
    ----------
    r : float
        Bond length in Angstrom.
    dtype : torch.dtype
        Floating-point precision (float32 for fast training).

    Returns
    -------
    AtomicGraph
        A fully populated graph with positions, edges, energy, and forces.
    """
    # --- Positions ---
    positions = torch.tensor(
        [[0.0, 0.0, 0.0],
         [r,   0.0, 0.0]],
        dtype=dtype,
    )  # (2, 3)

    # --- Atomic numbers (oxygen = 8) ---
    atomic_numbers = torch.tensor([Z_O, Z_O], dtype=torch.long)  # (2,)

    # --- Graph topology: bidirectional edges 0→1 and 1→0 ---
    #   edge_index[0] = source atoms ("row")
    #   edge_index[1] = target atoms ("col")
    edge_index = torch.tensor([[0, 1], [1, 0]], dtype=torch.long)  # (2, 2)

    # Edge displacement vectors: r_target − r_source
    edge_vectors = torch.tensor(
        [[ r, 0.0, 0.0],   # 0→1
         [-r, 0.0, 0.0]],  # 1→0
        dtype=dtype,
    )  # (2, 3)
    edge_lengths = torch.tensor([r, r], dtype=dtype)  # (2,)

    # --- Labels (energy and forces from Morse potential) ---
    r_np = np.array([r])
    E = float(morse_energy(r_np)[0])
    dE_dr = float(morse_force_magnitude(r_np)[0])

    # Force on atom 0: F0 = +dE/dr along x  (Newton's 3rd law convention)
    # Force on atom 1: F1 = -dE/dr along x
    # Note: force = -∂E/∂x, and ∂E/∂x₀ = -dE/dr, so F₀ = +dE/dr
    forces = torch.tensor(
        [[dE_dr, 0.0, 0.0],
         [-dE_dr, 0.0, 0.0]],
        dtype=dtype,
    )  # (2, 3)

    energy = torch.tensor([E], dtype=dtype)  # (1,)

    return AtomicGraph(
        positions=positions,
        atomic_numbers=atomic_numbers,
        cell=torch.zeros(3, 3, dtype=dtype),       # no periodic boundary
        pbc=torch.zeros(3, dtype=torch.bool),
        edge_index=edge_index,
        edge_vectors=edge_vectors,
        edge_lengths=edge_lengths,
        energy=energy,
        forces=forces,
    )


# Test with a single structure at equilibrium
g_test = make_o2_graph(r_e)
print("Test graph:")
print(f"  num_atoms    = {g_test.num_atoms}")
print(f"  positions    = {g_test.pos.tolist()}")
print(f"  edge_index   = {g_test.edge_index.tolist()}")
print(f"  edge_lengths = {g_test.edge_weight.tolist()}")
print(f"  energy       = {g_test.energy.item():.4f} eV")
print(f"  forces       = {g_test.forces.tolist()}")

### 3.2 Sample many structures and split into train / val / test

In [ ]:
N_TOTAL = 600

# Sample bond lengths: mix of uniform random + denser near equilibrium
rng = np.random.default_rng(0)
r_uniform  = rng.uniform(0.85, 3.5, size=int(0.7 * N_TOTAL))
r_near_eq  = rng.normal(r_e, 0.3, size=N_TOTAL - len(r_uniform))
r_near_eq  = np.clip(r_near_eq, 0.85, 3.5)
r_all      = np.concatenate([r_uniform, r_near_eq])
rng.shuffle(r_all)

all_graphs = [make_o2_graph(float(r)) for r in r_all]

n_train = int(0.70 * N_TOTAL)
n_val   = int(0.15 * N_TOTAL)

train_graphs = all_graphs[:n_train]
val_graphs   = all_graphs[n_train : n_train + n_val]
test_graphs  = all_graphs[n_train + n_val :]

print(f"Dataset split: {len(train_graphs)} train | {len(val_graphs)} val | {len(test_graphs)} test")

### 3.3 Create DataLoaders

PyTorch Geometric's `DataLoader` automatically batches multiple `AtomicGraph`
objects into a single large graph — atoms and edges are concatenated, and a
`batch` tensor records which atoms belong to which structure.

In [ ]:
from torch_geometric.loader import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_graphs,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_graphs,  batch_size=BATCH_SIZE, shuffle=False)

# Inspect a batch
sample_batch = next(iter(train_loader))
print("Sample batch:")
print(f"  num_graphs   = {sample_batch.num_graphs}")
print(f"  pos shape    = {sample_batch.pos.shape}   (N_total × 3)")
print(f"  edge shape   = {sample_batch.edge_index.shape}  (2 × E_total)")
print(f"  energy shape = {sample_batch.energy.shape}  (B,)")
print(f"  forces shape = {sample_batch.forces.shape} (N_total × 3)")
print(f"  batch tensor = {sample_batch.batch[:6].tolist()}...  (atom→graph assignment)")

---

## 4 · Model Design

### Why a pair-edge model?

For a diatomic molecule, the energy depends **only on the interatomic distance**
and the **types of the two atoms**.  The natural design is therefore an
**edge-centric MLP**: for each pair of atoms (i, j) we form a feature vector:

$$\mathbf{f}_{ij} = [\mathbf{h}_i \;\|\; \mathbf{h}_j \;\|\; \phi(r_{ij})]$$

where:
- $\mathbf{h}_i, \mathbf{h}_j$ — learnable atom-type embeddings
- $\phi(r_{ij})$ — **Bessel radial basis** expansion of the bond length

The MLP maps this to a scalar **edge energy**, and we sum edge energies to get
the total energy.  Forces follow by autograd.

```
z₀, z₁  ──[Embedding]──▶ h₀, h₁
                                   ┐
r₀₁     ──[BesselBasis]──▶ φ(r)  ─┤─ concat ─▶ MLP ─▶ E_edge ─▶ E_total
                                   ┘                               │
pos ◀── -∂E/∂pos ─────────────────────────────── autograd ────────┘
```

### Critical note: compute distances INSIDE `forward()`

To get correct autograd forces, distances must be **recomputed from `graph.pos`
inside the forward pass** — not read from the pre-stored `graph.edge_weight`.
Pre-stored values have no gradient connection to `graph.pos`.

```python
row, col = graph.edge_index
edge_vecs = graph.pos[col] - graph.pos[row]   # ← differentiable w.r.t. pos!
edge_lens = edge_vecs.norm(dim=-1)
```

---

## 5 · Approach 1: Monolithic `PairMLPModel`

A **monolithic model** takes an `AtomicGraph` and returns a prediction dict
directly — no backbone/head split.  This is the simplest way to plug a custom
architecture into GOAL.

The only requirement is:
- Inherits from `nn.Module`
- Has an `output_keys` property
- `forward(graph)` returns `dict[str, Tensor]`

In [ ]:
class PairMLPModel(nn.Module):
    """Monolithic MLP for diatomic pair energy + force prediction.

    Architecture
    ------------
    1. Atom-type embedding:  z  →  hᵢ ∈ ℝ^embed_dim
    2. Bessel basis:         rᵢⱼ → φ(r) ∈ ℝ^num_radial
    3. Edge feature:         [hᵢ ‖ hⱼ ‖ φ(r)] ∈ ℝ^(2·embed_dim + num_radial)
    4. Edge MLP:             f → E_edge ∈ ℝ
    5. Total energy:         E = Σ_edges E_edge / 2   (each pair counted twice)
    6. Forces:               F = −∂E/∂pos  (autograd)

    Parameters
    ----------
    num_elements : int
        Largest atomic number supported (120 covers the whole periodic table).
    embed_dim : int
        Dimension of the atomic-type embedding.
    hidden_dim : int
        Width of the hidden layers in the edge MLP.
    num_radial : int
        Number of Bessel basis functions.
    cutoff : float
        Cutoff radius (Å).  Edges longer than this carry zero signal.
    num_layers : int
        Number of hidden layers in the edge MLP.
    """

    def __init__(
        self,
        num_elements: int = 120,
        embed_dim: int = 16,
        hidden_dim: int = 64,
        num_radial: int = 8,
        cutoff: float = 5.0,
        num_layers: int = 3,
    ) -> None:
        super().__init__()

        # Atom-type embedding  (learned, not hand-crafted)
        self.embedding = AtomicNumberEmbedding(
            num_elements=num_elements,
            embedding_dim=embed_dim,
        )

        # Radial basis to encode bond length
        self.basis = BesselBasis(num_basis=num_radial, cutoff=cutoff)

        # Edge MLP: [hᵢ ‖ hⱼ ‖ φ(r)] → edge energy
        in_dim = 2 * embed_dim + num_radial
        layers: list[nn.Module] = [nn.Linear(in_dim, hidden_dim), nn.SiLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.SiLU()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.edge_mlp = nn.Sequential(*layers)

    # ------------------------------------------------------------------
    # MonolithicModel protocol (required for GOAL's training loop)
    # ------------------------------------------------------------------

    @property
    def output_keys(self) -> list[str]:
        return ["energy", "forces"]

    # ------------------------------------------------------------------
    # Forward pass
    # ------------------------------------------------------------------

    def forward(self, graph: AtomicGraph) -> dict[str, torch.Tensor]:
        """Predict energy and forces for a (batched) diatomic system.

        Parameters
        ----------
        graph : AtomicGraph
            Can be a single graph or a PyG Batch of multiple graphs.

        Returns
        -------
        dict with keys:
            'energy'    — (B,) total energy per structure
            'forces'    — (N, 3) atomic forces
            'num_atoms' — (B,) atom count per structure (for loss normalisation)
        """
        # ── Step 1: Enable gradient tracking on positions ──────────────────
        # This is essential for autograd forces.  We modify in-place so the
        # tensor's identity (and its place in the computation graph) stays
        # the same.
        graph.pos.requires_grad_(True)

        # ── Step 2: Atom-type embeddings ───────────────────────────────────
        h = self.embedding(graph.z)  # (N, embed_dim)

        # ── Step 3: Recompute distances from positions (DIFFERENTIABLE!) ───
        # IMPORTANT: we do NOT use graph.edge_weight (precomputed, no grad).
        # Recomputing here creates a path from energy → pos in the autograd
        # graph, which is required for non-zero forces.
        row, col = graph.edge_index                        # (E,), (E,)
        edge_vecs = graph.pos[col] - graph.pos[row]        # (E, 3)  ← has grad!
        edge_lens = edge_vecs.norm(dim=-1)                 # (E,)    ← has grad!

        # ── Step 4: Edge features ──────────────────────────────────────────
        phi = self.basis(edge_lens)                        # (E, num_radial)
        edge_feats = torch.cat([h[row], h[col], phi], dim=-1)  # (E, 2*D+num_radial)

        # ── Step 5: Per-edge energy contribution ───────────────────────────
        edge_energy = self.edge_mlp(edge_feats).squeeze(-1)  # (E,)

        # ── Step 6: Aggregate to per-graph energy ──────────────────────────
        # graph.batch maps each ATOM to a graph index.  For edges, we use the
        # source atom's graph index.
        batch = (
            graph.batch
            if graph.batch is not None
            else torch.zeros(graph.num_atoms, dtype=torch.long, device=graph.pos.device)
        )
        edge_batch = batch[row]  # (E,)
        num_graphs = int(batch.max().item()) + 1

        # Divide by 2: each pair (i,j) appears as two directed edges.
        energy = scatter(
            edge_energy, edge_batch, dim=0, reduce="sum", dim_size=num_graphs
        ) / 2.0  # (B,)

        # ── Step 7: Forces via autograd ────────────────────────────────────
        # F = −∂E/∂R  →  physics-consistent, energy-conserving forces
        grad = torch.autograd.grad(
            outputs=energy.sum(),
            inputs=graph.pos,
            create_graph=self.training,   # needed for 2nd-order optimisers
            retain_graph=True,
        )[0]                              # (N, 3)
        forces = -grad                    # (N, 3)

        # ── Step 8: Num-atoms per graph (for loss normalisation) ───────────
        num_atoms = scatter(
            torch.ones(graph.num_atoms, device=graph.pos.device),
            batch,
            dim=0,
            reduce="sum",
            dim_size=num_graphs,
        )  # (B,)

        return {"energy": energy, "forces": forces, "num_atoms": num_atoms}

In [ ]:
# ── Quick sanity check ────────────────────────────────────────────────────────
model_mono = PairMLPModel(
    embed_dim=16,
    hidden_dim=64,
    num_radial=8,
    cutoff=5.0,
    num_layers=3,
).to(device)

# Forward pass on a single graph
g_test = g_test.to(device)
model_mono.eval()
with torch.no_grad():
    # We still need requires_grad for forces, so use torch.enable_grad
    with torch.enable_grad():
        out = model_mono(g_test)

print("Output shapes (single graph):")
print(f"  energy    : {out['energy'].shape}  → {out['energy'].item():.4f} eV")
print(f"  forces    : {out['forces'].shape}")
print(f"  num_atoms : {out['num_atoms'].shape}  → {out['num_atoms'].item()}")
print()

n_params = sum(p.numel() for p in model_mono.parameters())
print(f"Total parameters: {n_params:,}")

---

## 6 · Approach 2: Custom `PairEnergyForcesHead`

GOAL's **modular** workflow separates the model into:

```
AtomicGraph → [Backbone] → NodeFeatures → [Head] → dict[str, Tensor]
```

A custom **head** must satisfy the `TaskHead` protocol:
- `forward(features: NodeFeatures, graph: AtomicGraph) → dict[str, Tensor]`
- `output_keys` property

Here we combine backbone **node features** with **edge geometry** to build an
edge-centric energy prediction, exactly like in the monolithic model but as a
re-usable, composable head.

This head can be paired with **any** invariant backbone (e.g. `InvariantGNN`).

In [ ]:
class PairEnergyForcesHead(nn.Module):
    """Custom head for pair energy + force prediction.

    Plugs into any invariant backbone that produces scalar node features
    (``NodeFeatures`` with ``irreps = 'Dx0e'``).

    For each directed edge (i→j) in the graph, it forms:

        f_ij = [h_i ‖ h_j ‖ BesselBasis(r_ij)]

    and maps this through an MLP to a scalar edge-energy contribution.
    The total energy is the sum of edge energies divided by 2, and forces
    are obtained via autograd.

    Parameters
    ----------
    backbone_hidden_dim : int
        Dimensionality of the node features produced by the backbone.
        Must match the backbone's output width.
    hidden_dim : int
        Width of the hidden layers in the edge MLP.
    num_radial : int
        Number of Bessel basis functions for distance encoding.
    cutoff : float
        Cutoff radius in Angstrom.  Should match the backbone's cutoff.
    num_layers : int
        Number of hidden layers in the edge MLP.
    """

    def __init__(
        self,
        backbone_hidden_dim: int,
        hidden_dim: int = 64,
        num_radial: int = 8,
        cutoff: float = 5.0,
        num_layers: int = 3,
    ) -> None:
        super().__init__()

        self.basis = BesselBasis(num_basis=num_radial, cutoff=cutoff)

        # Input: two node feature vectors + radial basis
        in_dim = 2 * backbone_hidden_dim + num_radial
        layers: list[nn.Module] = [nn.Linear(in_dim, hidden_dim), nn.SiLU()]
        for _ in range(num_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.SiLU()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.edge_mlp = nn.Sequential(*layers)

    @property
    def output_keys(self) -> list[str]:
        return ["energy", "forces", "num_atoms"]

    def forward(
        self,
        features: NodeFeatures,
        graph: AtomicGraph,
    ) -> dict[str, torch.Tensor]:
        """Compute energy and forces from backbone node features.

        Parameters
        ----------
        features : NodeFeatures
            Output of the backbone; provides ``node_feats`` (N, D).
        graph : AtomicGraph
            The input graph (provides positions and edge topology).

        Returns
        -------
        dict with 'energy' (B,), 'forces' (N, 3), 'num_atoms' (B,).
        """
        # Enable gradient on positions for force computation
        graph.pos.requires_grad_(True)

        # Recompute edge lengths from positions (MUST be inside forward!)
        row, col = graph.edge_index
        edge_vecs = graph.pos[col] - graph.pos[row]    # (E, 3)  differentiable
        edge_lens = edge_vecs.norm(dim=-1)              # (E,)    differentiable

        # Node features from backbone
        h = features.node_feats   # (N, backbone_hidden_dim)

        # Build edge features
        phi = self.basis(edge_lens)                             # (E, num_radial)
        edge_feats = torch.cat([h[row], h[col], phi], dim=-1)  # (E, in_dim)

        # Edge energy
        edge_energy = self.edge_mlp(edge_feats).squeeze(-1)    # (E,)

        # Aggregate to per-graph energy
        batch = (
            graph.batch
            if graph.batch is not None
            else torch.zeros(graph.num_atoms, dtype=torch.long, device=graph.pos.device)
        )
        edge_batch = batch[row]
        num_graphs = int(batch.max().item()) + 1

        energy = scatter(
            edge_energy, edge_batch, dim=0, reduce="sum", dim_size=num_graphs
        ) / 2.0  # (B,)

        # Forces via autograd
        grad = torch.autograd.grad(
            outputs=energy.sum(),
            inputs=graph.pos,
            create_graph=self.training,
            retain_graph=True,
        )[0]
        forces = -grad  # (N, 3)

        # Num-atoms per graph
        num_atoms = scatter(
            torch.ones(graph.num_atoms, device=graph.pos.device),
            batch,
            dim=0,
            reduce="sum",
            dim_size=num_graphs,
        )  # (B,)

        return {"energy": energy, "forces": forces, "num_atoms": num_atoms}

In [ ]:
# ── Wire a backbone + our custom head ────────────────────────────────────────
from goal.ml.nn.models.invariant import InvariantGNN

HIDDEN_DIM = 64
CUTOFF     = 5.0

backbone = InvariantGNN(
    num_elements=120,
    embedding_dim=32,
    hidden_channels=HIDDEN_DIM,
    num_interactions=2,
    cutoff=CUTOFF,
    num_radial_basis=8,
).to(device)

pair_head = PairEnergyForcesHead(
    backbone_hidden_dim=HIDDEN_DIM,
    hidden_dim=64,
    num_radial=8,
    cutoff=CUTOFF,
    num_layers=2,
).to(device)

# Sanity check: backbone → head pipeline
g_test = make_o2_graph(r_e).to(device)
with torch.enable_grad():
    node_features = backbone(g_test)           # NodeFeatures
    out_modular   = pair_head(node_features, g_test)

print("Modular (InvariantGNN + PairHead) output:")
print(f"  energy    : {out_modular['energy'].item():.4f} eV")
print(f"  forces    : {out_modular['forces'].tolist()}")
print()
total_params = (
    sum(p.numel() for p in backbone.parameters())
    + sum(p.numel() for p in pair_head.parameters())
)
print(f"Total parameters (backbone + head): {total_params:,}")

---

## 7 · Training

### 7.1 Loss function

GOAL's `CompositeLoss` lets you mix multiple weighted losses:

```
L_total = w_E × L_energy + w_F × L_forces
```

`EnergyLoss` divides by the number of atoms (per-atom energy MAE),
which makes the loss scale-invariant across system sizes.

We use a **higher weight for forces** because:
1. Forces provide more training signal (3N values vs 1 energy per structure)
2. Getting forces right is crucial for MD simulation stability

In [ ]:
def build_loss(energy_weight: float = 1.0, forces_weight: float = 10.0) -> CompositeLoss:
    """Build a composite energy + forces loss."""
    return CompositeLoss([
        WeightedLoss(EnergyLoss(loss_fn="mae"),  weight=energy_weight, label="energy_loss"),
        WeightedLoss(ForcesLoss(loss_fn="mae"),  weight=forces_weight, label="forces_loss"),
    ])


# Helper: extract a target dict from an AtomicGraph batch
# CompositeLoss expects dict[str, Tensor] for both predictions and targets.
def graph_to_target_dict(batch: AtomicGraph) -> dict[str, torch.Tensor]:
    """Convert an AtomicGraph batch into the target dict expected by the losses."""
    batch_vec = (
        batch.batch
        if batch.batch is not None
        else torch.zeros(batch.num_atoms, dtype=torch.long, device=batch.pos.device)
    )
    num_graphs = int(batch_vec.max().item()) + 1
    num_atoms_per_graph = scatter(
        torch.ones(batch.num_atoms, device=batch.pos.device),
        batch_vec,
        dim=0,
        reduce="sum",
        dim_size=num_graphs,
    )
    return {
        "energy":    batch.energy,       # (B,)
        "forces":    batch.forces,       # (N, 3)
        "num_atoms": num_atoms_per_graph, # (B,)  ← required by EnergyLoss
    }

### 7.2 Training loop

We write a minimal but complete training loop.  For production training
(multi-GPU, callbacks, logging, checkpointing) you would use
`GOALModule` + PyTorch Lightning instead.

In [ ]:
def train_model(
    model: nn.Module,
    train_loader,
    val_loader,
    *,
    num_epochs: int = 60,
    lr: float = 1e-3,
    energy_weight: float = 1.0,
    forces_weight: float = 10.0,
    device: torch.device = torch.device("cpu"),
) -> dict:
    """Train a GOAL model and return loss history.

    Works with any model whose forward(graph) → dict[str, Tensor].

    Parameters
    ----------
    model        : the model to train (moved to device inside)
    train_loader : PyG DataLoader for training data
    val_loader   : PyG DataLoader for validation data
    num_epochs   : training epochs
    lr           : AdamW learning rate
    energy_weight: weight for the energy loss term
    forces_weight: weight for the forces loss term
    device       : torch device

    Returns
    -------
    dict with 'train_total', 'val_total', 'train_energy', etc. per epoch
    """
    model = model.to(device)
    criterion = build_loss(energy_weight, forces_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs, eta_min=lr * 0.01
    )

    history = {"train_total": [], "val_total": [], "train_energy": [], "train_forces": []}

    for epoch in range(num_epochs):
        # ── Training ──────────────────────────────────────────────────────
        model.train()
        train_losses: dict[str, float] = {}

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad(set_to_none=True)

            predictions = model(batch)
            targets     = graph_to_target_dict(batch)
            losses      = criterion(predictions, targets)

            losses["total"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            for k, v in losses.items():
                train_losses[k] = train_losses.get(k, 0.0) + v.item()

        n_batches = len(train_loader)
        for k in train_losses:
            train_losses[k] /= n_batches

        # ── Validation ────────────────────────────────────────────────────
        model.eval()
        val_total = 0.0

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                with torch.enable_grad():   # needed for autograd forces
                    predictions = model(batch)
                targets = graph_to_target_dict(batch)
                losses  = criterion(predictions, targets)
                val_total += losses["total"].item()

        val_total /= len(val_loader)
        scheduler.step()

        # ── Record history ────────────────────────────────────────────────
        history["train_total"].append(train_losses["total"])
        history["val_total"].append(val_total)
        history["train_energy"].append(train_losses.get("energy_loss", 0.0))
        history["train_forces"].append(train_losses.get("forces_loss", 0.0))

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(
                f"Epoch {epoch+1:>3}/{num_epochs}  "
                f"train={train_losses['total']:.4f}  "
                f"val={val_total:.4f}  "
                f"E={train_losses.get('energy_loss', 0):.4f}  "
                f"F={train_losses.get('forces_loss', 0):.4f}  "
                f"lr={scheduler.get_last_lr()[0]:.2e}"
            )

    return history

### 7.3 Train the monolithic model

In [ ]:
model_mono = PairMLPModel(
    embed_dim=16,
    hidden_dim=64,
    num_radial=8,
    cutoff=5.0,
    num_layers=3,
)

print("Training PairMLPModel (monolithic)...")
history_mono = train_model(
    model_mono,
    train_loader,
    val_loader,
    num_epochs=80,
    lr=5e-3,
    energy_weight=1.0,
    forces_weight=10.0,
    device=device,
)

### 7.4 Train the modular model (InvariantGNN + PairEnergyForcesHead)

We wrap the backbone + head pair in a thin `nn.Module` so the training loop
stays identical.

In [ ]:
class ModularPairModel(nn.Module):
    """Thin wrapper that runs backbone → head and exposes a single forward()."""

    def __init__(self, backbone: nn.Module, head: nn.Module) -> None:
        super().__init__()
        self.backbone = backbone
        self.head = head

    @property
    def output_keys(self) -> list[str]:
        return self.head.output_keys

    def forward(self, graph: AtomicGraph) -> dict[str, torch.Tensor]:
        features = self.backbone(graph)
        return self.head(features, graph)


HIDDEN_DIM = 64
CUTOFF     = 5.0

backbone_mod = InvariantGNN(
    num_elements=120,
    embedding_dim=32,
    hidden_channels=HIDDEN_DIM,
    num_interactions=2,
    cutoff=CUTOFF,
    num_radial_basis=8,
)
head_mod = PairEnergyForcesHead(
    backbone_hidden_dim=HIDDEN_DIM,
    hidden_dim=64,
    num_radial=8,
    cutoff=CUTOFF,
    num_layers=2,
)
model_modular = ModularPairModel(backbone_mod, head_mod)

print("Training modular model (InvariantGNN + PairEnergyForcesHead)...")
history_mod = train_model(
    model_modular,
    train_loader,
    val_loader,
    num_epochs=80,
    lr=5e-3,
    energy_weight=1.0,
    forces_weight=10.0,
    device=device,
)

### 7.5 Plot training curves

In [ ]:
epochs = range(1, len(history_mono["train_total"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Total loss
axes[0].plot(epochs, history_mono["train_total"], "C0",   label="Mono — train")
axes[0].plot(epochs, history_mono["val_total"],   "C0--", label="Mono — val")
axes[0].plot(epochs, history_mod["train_total"],  "C1",   label="Modular — train")
axes[0].plot(epochs, history_mod["val_total"],    "C1--", label="Modular — val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Composite loss")
axes[0].set_title("Total loss (energy + forces)")
axes[0].legend(fontsize=8)
axes[0].set_yscale("log")
axes[0].grid(alpha=0.3)

# Energy vs forces loss
axes[1].plot(epochs, history_mono["train_energy"], "C0",   lw=1.5, label="Mono energy")
axes[1].plot(epochs, history_mono["train_forces"], "C0--", lw=1.5, label="Mono forces")
axes[1].plot(epochs, history_mod["train_energy"],  "C1",   lw=1.5, label="Modular energy")
axes[1].plot(epochs, history_mod["train_forces"],  "C1--", lw=1.5, label="Modular forces")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Weighted loss component")
axes[1].set_title("Energy vs forces loss")
axes[1].legend(fontsize=8)
axes[1].set_yscale("log")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 8 · Evaluation: Predicted Energy & Force Curves

We scan bond length from 0.85 to 4.0 Å and compare the model's predictions
to the analytic Morse potential.

In [ ]:
@torch.no_grad()
def predict_curve(
    model: nn.Module,
    r_values: np.ndarray,
    device: torch.device,
) -> tuple[np.ndarray, np.ndarray]:
    """Run the model at each bond length and return (energies, force_x_atom0).

    We use torch.enable_grad() inside because autograd forces require it.
    """
    model.eval()
    energies, forces_x = [], []

    for r in r_values:
        g = make_o2_graph(float(r)).to(device)
        with torch.enable_grad():
            out = model(g)
        energies.append(out["energy"].item())
        forces_x.append(out["forces"][0, 0].item())   # x-component of force on atom 0

    return np.array(energies), np.array(forces_x)


r_eval    = np.linspace(0.85, 4.0, 200)
E_true    = morse_energy(r_eval)
F_true    = morse_force_magnitude(r_eval)     # dE/dr  (= force on atom 0 in x)

model_mono    = model_mono.to(device)
model_modular = model_modular.to(device)

E_mono, F_mono = predict_curve(model_mono,    r_eval, device)
E_mod,  F_mod  = predict_curve(model_modular, r_eval, device)

print("Evaluation complete.")
print(f"  Mono    energy MAE : {np.mean(np.abs(E_mono - E_true)):.4f} eV")
print(f"  Modular energy MAE : {np.mean(np.abs(E_mod  - E_true)):.4f} eV")
print(f"  Mono    force  MAE : {np.mean(np.abs(F_mono - F_true)):.4f} eV/Å")
print(f"  Modular force  MAE : {np.mean(np.abs(F_mod  - F_true)):.4f} eV/Å")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Energy curves ────────────────────────────────────────────────────────────
axes[0].plot(r_eval, E_true, "k",    lw=2.5, label="Morse (true)", zorder=3)
axes[0].plot(r_eval, E_mono, "C0--", lw=1.8, label="PairMLPModel (monolithic)")
axes[0].plot(r_eval, E_mod,  "C1:",  lw=1.8, label="InvariantGNN + PairHead")
axes[0].axvline(r_e, ls="--", color="gray", alpha=0.7, label=f"$r_e={r_e}$ Å")
axes[0].scatter(
    [float(r_all[i]) for i in range(0, 20)],
    [float(morse_energy(np.array([r_all[i]]))[0]) for i in range(0, 20)],
    s=15, c="C3", alpha=0.5, zorder=2, label="Training points (sample)"
)
axes[0].set_xlabel("Bond length $r$ (Å)", fontsize=12)
axes[0].set_ylabel("Energy (eV)", fontsize=12)
axes[0].set_title("Energy Curve: Predicted vs True", fontsize=13)
axes[0].set_ylim(-7, 5)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# ── Force curves ─────────────────────────────────────────────────────────────
axes[1].plot(r_eval, F_true, "k",    lw=2.5, label="Morse (true)", zorder=3)
axes[1].plot(r_eval, F_mono, "C0--", lw=1.8, label="PairMLPModel (monolithic)")
axes[1].plot(r_eval, F_mod,  "C1:",  lw=1.8, label="InvariantGNN + PairHead")
axes[1].axvline(r_e, ls="--", color="gray", alpha=0.7)
axes[1].axhline(0,   ls="-",  color="k",    lw=0.8)
axes[1].set_xlabel("Bond length $r$ (Å)", fontsize=12)
axes[1].set_ylabel("Force $dE/dr$ on atom 0 (eV/Å)", fontsize=12)
axes[1].set_title("Force Curve: Predicted vs True", fontsize=13)
axes[1].set_ylim(-20, 20)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 9 · Bonus: Register the Custom Head with GOAL's Registry

For production use, you can **register** your custom head so it is
accessible by name from Hydra configs, just like the built-in heads.

```yaml
# configs/model/my_pair_model.yaml
model:
  backbone:
    name: invariant_gnn
    hidden_channels: 64
    ...
  head:
    name: pair_energy_forces   # ← your registered name
    backbone_hidden_dim: 64
    hidden_dim: 64
    num_radial: 8
    cutoff: 5.0
```

Register like this:

In [ ]:
from goal.ml.registry import HEAD_REGISTRY

# Register the head under the name 'pair_energy_forces'
HEAD_REGISTRY.register_instance("pair_energy_forces", PairEnergyForcesHead)

# Verify it's accessible
retrieved_cls = HEAD_REGISTRY.get("pair_energy_forces")
print(f"Retrieved class: {retrieved_cls}")

# Instantiate via registry (as Hydra would)
head_from_registry = HEAD_REGISTRY.build(
    "pair_energy_forces",
    backbone_hidden_dim=64,
    hidden_dim=64,
    num_radial=8,
    cutoff=5.0,
)
print(f"Instantiated: {head_from_registry}")

---

## 10 · Summary and Next Steps

### What we built

| Component | Class | GOAL protocol |
|---|---|---|
| Monolithic model | `PairMLPModel` | `MonolithicModel` (no head) |
| Custom head | `PairEnergyForcesHead` | `TaskHead` |
| Loss | `CompositeLoss` | energy + forces |

### Key takeaways

1. **Recompute distances inside `forward()`** — not from pre-stored `graph.edge_weight`.  
   This creates the gradient path needed for autograd forces.

2. **Divide edge energies by 2** — because GOAL graphs are bidirectional
   (each atom pair has two directed edges).

3. **Forces are free** once energy is differentiable — `F = −∂E/∂pos` via
   `torch.autograd.grad` guarantees energy conservation.

4. **Use `create_graph=self.training`** in autograd — required for second-order
   gradients (and force loss backprop).

### Next steps

- **Real dataset**: Replace the Morse potential with actual DFT data  
  (see `examples/datasets/` for loaders for ANI-1, MD17, QM9, SPICE)
- **Larger systems**: The same `PairEnergyForcesHead` works for any molecule  
  — just use a cutoff that covers relevant interactions
- **Equivariance**: Swap `InvariantGNN` for `HyperSpecModel` to get  
  equivariant node features (better data efficiency, especially for forces)
- **Production training**: Use `GOALModule` + PyTorch Lightning for  
  multi-GPU training, EMA, checkpointing, and Hydra-based hyperparameter tuning